**Model Training in LLM**

*   Set vocab_size=30000
*   hidden_size=768
*   is_decoder=True

Lets you keep your exact (8, 30k) ->(8, 768)  dimensions without fighting BERT's bidirectional pretrained weights.




### First Step: Tokenization


> Note: Training input = "The bank of the river flooded"



In [1]:
import torch
from transformers import AutoTokenizer, BertConfig, BertLMHeadModel

sentence = "The bank of the river flooded"
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
# 1. Configure BERT as a causal decoder to block forward lookup
config = BertConfig.from_pretrained("bert-base-uncased")
config.is_decoder = True  # Enforces causal (left-to-right) masking
# 2. Load with a causal language modeling head (or BertModel(config) for hidden states only)
model = BertLMHeadModel.from_pretrained("bert-base-uncased", config=config)
inputs = tokenizer(sentence, max_length=8, padding="max_length", truncation=True, return_tensors="pt")
print(inputs)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

[transformers] BertLMHeadModel LOAD REPORT from: bert-base-uncased
Key                         | Status     |  | 
----------------------------+------------+--+-
bert.pooler.dense.bias      | UNEXPECTED |  | 
bert.pooler.dense.weight    | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


{'input_ids': tensor([[  101,  1996,  2924,  1997,  1996,  2314, 10361,   102]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1]])}


### Second Step: Text Embedding


In [4]:
# Passing labels automatically shifts them right so token i predicts token i+1 without leakage
outputs = model(**inputs, labels=inputs["input_ids"])
loss = outputs.loss
logits = outputs.logits  # Shape: (1, 8, 30522)

In [5]:
x.shape

torch.Size([1, 8, 30522])

### Extracting the 768-Dimensional Hidden Embeddings
To extract the raw representations before they are projected to the vocabulary size (30,522 logits), we can pass `output_hidden_states=True` to our model, or use the base `model.bert` attribute to query the hidden layers directly.

In [9]:
# Pass outputs through the model and request hidden states
outputs_with_hidden = model(**inputs, output_hidden_states=True)

# The last hidden state represents the final layer embeddings of shape [Batch, Sequence_Length, Hidden_Size]
last_hidden_state = outputs_with_hidden.hidden_states[-1]
print(f"Last Hidden State Shape (Embeddings): {last_hidden_state.shape}") # Should be [1, 8, 768]

# Let's assign this 768-dim tensor to x to use with our Multi-Head Attention layer
x_embeddings = last_hidden_state

Last Hidden State Shape (Embeddings): torch.Size([1, 8, 768])


### Processing Real Embeddings with Multi-Head Attention
Now, let's feed our 768-dimensional token embeddings (`x_embeddings`) into our custom `MultiHeadAttention` layer using 12 heads (each head with dimension $768 / 12 = 64$).

In [7]:
# Multi-head Attention
# B: Batch Size (e.g., 1 sequence)
# T: Sequence Length / Time Steps (e.g., 8 tokens)
# D: Feature Dimension / Hidden Size / Vocab Size (e.g., 30522 logits)
import torch
import torch.nn as nn
import torch.nn.functional as F

class MultiHeadAttention(nn.Module):
  def __init__(self, d_model:int, num_heads:int, dropout: float = 0.0):
    super().__init__()
    self.d_model = d_model
    self.num_heads = num_heads
    self.d_k = d_model // num_heads
    self.head_dim = self.d_k  # Add alias to prevent AttributeError in any version

    # (QKt/sq(model))* V = (d_model, d_model)
    self.qkv_proj = nn.Linear(d_model, 3 * d_model)
    self.out_proj = nn.Linear(d_model, d_model)
    self.dropout = nn.Dropout(dropout)

  def forward(self, x:torch.Tensor, is_causal: bool = False)->torch.Tensor:
    B, T, D = x.shape
    qkv = self.qkv_proj(x)
    q,k,v = qkv.chunk(3, dim=-1)
    q,k,v = map(lambda t: t.view(B, T, self.num_heads, self.d_k).transpose(1,2), (q,k,v))
    attention_pattern = torch.matmul(q, k.transpose(-2, -1)) / (self.d_k ** 0.5)

    # Apply causal mask only if is_causal is True
    if is_causal:
      causal_mask = torch.tril(torch.ones(T, T, device=x.device, dtype=torch.bool))
      attention_pattern = attention_pattern.masked_fill(~causal_mask, float('-inf'))

    # 5. Softmax along key sequence dimension
    attn_weights = F.softmax(attention_pattern, dim=-1)
    attn_weights = self.dropout(attn_weights)
    # 6. Weighted sum over values: (B, H, T, T) @ (B, H, T, d_k) -> (B, H, T, d_k)
    context = torch.matmul(attn_weights, v)
    context = context.transpose(1, 2).contiguous().view(B, T, D)

    # 8. Final linear projection
    return self.out_proj(context)

In [10]:
# Initialize MHA for d_model=768 and num_heads=12
mha_768 = MultiHeadAttention(d_model=768, num_heads=12)

# Run causal self-attention on our actual text embeddings
attention_output = mha_768(x_embeddings, is_causal=True)

print(f"Input shape: {x_embeddings.shape}")
print(f"Attention Output shape: {attention_output.shape}")

Input shape: torch.Size([1, 8, 768])
Attention Output shape: torch.Size([1, 8, 768])
